# SFT with TRL and PEFT

Use this in a Python 3.11 Jupyter environment with the tool requirements installed and the working directory set to the repository root. This plain PyTorch path explains the training pieces. It uses full-precision base weights with a small LoRA adapter. An NVIDIA GPU is recommended; CPU training is slow. The evaluation split is never loaded into the trainer.

In [ ]:
import sys
from pathlib import Path
assert Path('configs/models.yaml').exists(), 'Start Jupyter from the repository root.'
sys.path.insert(0, 'scripts')
from common import model_id
from configure_chat import configure_tokenizer
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
base = model_id('starter')

## Load the base and original chat template

The tokenizer formats roles and turn boundaries exactly as the model expects. Training on the whole short conversation is intentional here; assistant-only masking requires a compatible template.

In [ ]:
tokenizer = configure_tokenizer(AutoTokenizer.from_pretrained(base))
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float32)
data = load_dataset('json', data_files={s: f'recipes/json-extractor/data/{s}.jsonl' for s in ['train','valid']})
def format_chat(batch):
    return {'text': [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False, enable_thinking=False) for m in batch['messages']]}
data = data.map(format_chat, batched=True, remove_columns=['messages'])

## Train the adapter

LoRA learns a small set of extra weights. Gradient accumulation increases the effective batch without placing all examples in memory at once.

In [ ]:
config = SFTConfig(output_dir='outputs/trl/checkpoints', per_device_train_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=2e-4, num_train_epochs=3,
    max_length=1024, dataset_text_field='text', report_to='none', seed=42,
    eval_strategy='epoch', save_strategy='epoch', logging_steps=1, packing=False,
    use_cpu=not torch.cuda.is_available(), bf16=False, fp16=False)
trainer = SFTTrainer(model=model, processing_class=tokenizer,
    train_dataset=data['train'], eval_dataset=data['valid'], args=config,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear',
        lora_dropout=0, task_type='CAUSAL_LM'))
trainer.train()
trainer.save_model('outputs/trl/adapter')
tokenizer.save_pretrained('outputs/trl/adapter')

## Merge for export

Keep the adapter as well as the merged model. Merge into unquantized base weights before converting to GGUF.

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained('outputs/trl/merged', safe_serialization=True)
tokenizer.save_pretrained('outputs/trl/merged')

Next: guides/04-evaluate.md for a base-versus-tuned comparison, guides/05-quantize.md for GGUF, and guides/06-run-locally.md for inference. Record actual elapsed time, model revision and hardware. No outputs are shipped in this notebook.